# 03: Model Training (Transfer Learning from AI4Bharat/INCLUDE)
This notebook trains a classifier for 11 ISL gestures by fine-tuning the pretrained weights released by AI4Bharat for their 263-class INCLUDE dataset.

**Important Data Note:** The AI4Bharat pre-trained models expect 134-dimensional 2D inputs (MediaPipe x,y coordinates). Our original pipeline uses 258-dimensional 3D inputs (x,y,z,visibility). We will first run AI4Bharat's `generate_keypoints.py` against our raw `.mp4` videos to extract compatible 2D features before training.

In [ ]:
# @title 1. Setup Environment
!pip install timm xgboost mediapipe transformers scikit-learn
!git clone https://github.com/AI4Bharat/INCLUDE.git
%cd INCLUDE
# Note: we disable xgboost gpu_hist for compatibility
!sed -i 's/gpu_hist/hist/g' configs.py

### Upload Raw Data
You must upload your `data/raw` folder containing the 193 `.mp4` clips to the Colab environment. Place them in a folder named `raw_videos` with subfolders for each class (e.g., `raw_videos/Greetings/Hello_MVI_0037.mp4`).

In [ ]:
# @title 2. Extract 2D Keypoints (AI4Bharat Format)
import os
import json
import pandas as pd

# The raw videos should be at ../raw_videos, which corresponds to our 11 classes
raw_dir = "../raw_videos"
keypoints_dir = "isl_11_train_keypoints"
val_keypoints_dir = "isl_11_val_keypoints"
test_keypoints_dir = "isl_11_test_keypoints"

if not os.path.exists(raw_dir):
    print(f"Please upload your raw videos to {raw_dir}")
else:
    # Run the official feature extractor
    !python generate_keypoints.py --data_dir {raw_dir} --save_path isl_all_keypoints

    # Build Label Map
    classes = sorted([d for d in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, d))])
    label_map = {cls: idx for idx, cls in enumerate(classes)}
    os.makedirs("label_maps", exist_ok=True)
    with open("label_maps/isl_11.json", "w") as f:
        json.dump(label_map, f)
    print("Classes mapped:", label_map)

In [ ]:
# @title 3. Split Dataset Matching Our Baseline
import shutil
import numpy as np

# In this cell, you should upload your data/splits/labels.csv 
# to ensure we use the EXACT same 70/15/15 stratified split as the from-scratch baseline.
labels_csv_path = "../labels.csv"

if not os.path.exists(labels_csv_path):
    print(f"Please upload your labels.csv to {labels_csv_path}")
else:
    df = pd.read_csv(labels_csv_path)
    
    os.makedirs(keypoints_dir, exist_ok=True)
    os.makedirs(val_keypoints_dir, exist_ok=True)
    os.makedirs(test_keypoints_dir, exist_ok=True)
    
    for _, row in df.iterrows():
        # labels.csv uses video filenames like Greetings/Hello_MVI_0037.mp4
        # AI4Bharat's generate_keypoints.py creates json files named like Hello_MVI_0037.json
        rel_path = row['video_path']
        class_name = rel_path.split('/')[0]
        basename = os.path.splitext(os.path.basename(rel_path))[0]
        json_name = f"{basename}.json"
        
        src_path = os.path.join("isl_all_keypoints", class_name, json_name)
        
        if not os.path.exists(src_path):
            continue
            
        split = row['split']
        dst_dir = keypoints_dir if split == 'train' else val_keypoints_dir if split == 'val' else test_keypoints_dir
        
        os.makedirs(os.path.join(dst_dir, class_name), exist_ok=True)
        shutil.copy(src_path, os.path.join(dst_dir, class_name, json_name))
        
    print("Dataset split complete.")

In [ ]:
# @title 4. Prepare Custom Training Script
%%writefile train_transfer.py
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from models import Transformer, LSTM
from configs import TransformerConfig, LstmConfig
from dataset import KeypointsDataset
from utils import load_label_map
import json
import urllib.request
from sklearn.metrics import classification_report

def modify_model_head(model, num_classes=11):
    # Depending on LSTM or Transformer, the final layer might be named differently
    if hasattr(model, 'fc'):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif hasattr(model, 'classifier'):
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
    return model

def download_and_load_checkpoint(model_type, model, num_classes=11):
    if model_type == 'transformer':
        url = "https://api.wandb.ai/files/abdur-ai4bharat/include50-no-cnn/11d20bb9/augs_transformer.pth"
        filename = "include50_no_cnn_transformer_small.pth"
    else:
        url = "https://api.wandb.ai/files/abdur-ai4bharat/include-no-cnn/2prih6pi/augs_lstm.pth"
        filename = "include_no_cnn_lstm.pth"
        
    if not os.path.exists(filename):
        print(f"Downloading pretrained {model_type}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(filename, 'wb') as out:
            out.write(response.read())
            
    print("Loading state dict with strict=False...")
    ckpt = torch.load(filename, map_location='cpu', weights_only=False)
    
    # Filter out the final layer from the checkpoint because shape mismatches [50, hidden] vs [11, hidden]
    state_dict = ckpt['model']
    state_dict = {k: v for k, v in state_dict.items() if 'fc' not in k and 'classifier' not in k}
    
    model.load_state_dict(state_dict, strict=False)
    model = modify_model_head(model, num_classes)
    return model

def run_fine_tuning(model_type='transformer', epochs_warmup=15, epochs_finetune=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    label_map = load_label_map("isl_11")
    
    # Dataloaders
    train_dataset = KeypointsDataset("isl_11_train_keypoints", use_augs=True, label_map=label_map, mode="train", max_frame_len=169)
    val_dataset = KeypointsDataset("isl_11_val_keypoints", use_augs=False, label_map=label_map, mode="val", max_frame_len=169)
    test_dataset = KeypointsDataset("isl_11_test_keypoints", use_augs=False, label_map=label_map, mode="test", max_frame_len=169)
    
    train_dl = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_dl = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_dl = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    # Model
    if model_type == 'transformer':
        # AI4Bharat uses 256 for include50 pretrained models in their train_nn.py
        config = TransformerConfig(size='small', max_position_embeddings=256)
        model = Transformer(config=config, n_classes=50) # start with 50 so it matches the config initially
    else:
        config = LstmConfig()
        model = LSTM(config=config, n_classes=263)
        
    model = download_and_load_checkpoint(model_type, model, len(label_map))
    model = model.to(device)
    
    # 1. Warmup: freeze encoder, train only head
    for name, param in model.named_parameters():
        if 'fc' not in name and 'classifier' not in name:
            param.requires_grad = False
            
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=0.01)
    
    print("--- Phase 1: Linear Probing (Warmup) ---")
    for epoch in range(epochs_warmup):
        model.train()
        for batch in train_dl:
            x, y = batch['data'].to(device), batch['label'].to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = torch.nn.functional.cross_entropy(out, y)
            loss.backward()
            optimizer.step()
        print(f"Warmup Epoch {epoch+1}/{epochs_warmup} | Loss: {loss.item():.4f}")
        
    # 2. Fine-tuning: unfreeze all
    print("--- Phase 2: Full Fine-tuning ---")
    for param in model.parameters():
        param.requires_grad = True
        
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    
    best_val_acc = 0.0
    
    for epoch in range(epochs_finetune):
        model.train()
        for batch in train_dl:
            x, y = batch['data'].to(device), batch['label'].to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = torch.nn.functional.cross_entropy(out, y)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_dl:
                x, y = batch['data'].to(device), batch['label'].to(device)
                out = model(x)
                preds = torch.argmax(out, dim=-1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total
        print(f"Finetune Epoch {epoch+1}/{epochs_finetune} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({'model': model.state_dict(), 'val_acc': val_acc}, f"best_{model_type}_transfer.pth")
            
    # 3. Test Evaluation
    print("--- Test Evaluation ---")
    model.load_state_dict(torch.load(f"best_{model_type}_transfer.pth", weights_only=False)['model'])
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_dl:
            x, y = batch['data'].to(device), batch['label'].to(device)
            out = model(x)
            preds = torch.argmax(out, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            
    inv_label_map = {v: k for k, v in label_map.items()}
    target_names = [inv_label_map[i] for i in range(len(label_map))]
    
    print(classification_report(all_labels, all_preds, target_names=target_names))

if __name__ == "__main__":
    print("Running Transformer Transfer Learning...")
    run_fine_tuning(model_type='transformer', epochs_warmup=10, epochs_finetune=15)
    
    print("\n\nRunning BiLSTM Transfer Learning...")
    run_fine_tuning(model_type='lstm', epochs_warmup=10, epochs_finetune=15)

In [ ]:
# @title 5. Execute Training
!python train_transfer.py